# LeetCode 816 — Ambiguous Coordinates bằng Python

Link đề: https://leetcode.com/problems/ambiguous-coordinates/ · Độ khó: **Medium** · Chủ đề: String, Backtracking (quay lui), Enumeration (liệt kê)

Skeleton LeetCode:

```python
class Solution:
    def ambiguousCoordinates(self, s: str) -> List[str]:
        ...
```

Ghi chú: bài **liệt kê** mọi cách khôi phục tọa độ từ chuỗi chữ số đã bị xóa dấu phẩy, dấu chấm thập phân và khoảng trắng. Thứ tự kết quả **không quan trọng** (đề cho phép trả theo thứ tự bất kỳ) nên khi kiểm tra ta **sắp xếp rồi so sánh**. Điểm mấu chốt là quy tắc **số hợp lệ**: không có số 0 thừa ở đầu phần nguyên và không có số 0 thừa ở cuối phần thập phân.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: từ `"(123)"` khôi phục mọi tọa độ `"(x, y)"` có thể có, trong đó `x`, `y` là số hợp lệ (không có số 0 thừa, không có dấu `.` đứng đầu).
2. Nắm các công cụ Python cần thiết: cắt chuỗi, `startswith`/`endswith`, vòng lặp lồng nhau trong list comprehension (tích Descartes), `re.fullmatch`, `decimal.Decimal`, f-string, và `sorted` để so sánh kết quả không phụ thuộc thứ tự.
3. Tách bài thành **hai quyết định độc lập**: (a) chọn vị trí đặt dấu phẩy; (b) với mỗi vế, chọn đặt dấu chấm thập phân ở đâu (hoặc không). Hai vế độc lập nên ta **liệt kê số hợp lệ của từng vế rồi ghép cặp**.
4. Đi qua 5 cách: liệt kê thô rồi lọc bằng `Decimal`, tách theo vị trí chấm với luật đầu/cuối, phân tích trường hợp (xây trực tiếp), liệt kê rồi lọc bằng regex, quay lui từng ký tự có cắt tỉa.
5. Tránh các lỗi kinh điển: bỏ sót luật số 0 đầu hoặc cuối, đặt dấu phẩy ở đầu/cuối, quên khoảng trắng sau dấu phẩy, quên trường hợp không có dấu chấm, và so sánh kết quả theo thứ tự thay vì theo tập.

## 2. Đề bài & ví dụ

Ta từng có các tọa độ 2 chiều như `"(1, 3)"` hoặc `"(2, 0.5)"`. Sau đó ta xóa **mọi dấu phẩy, dấu chấm thập phân và khoảng trắng**, còn lại chuỗi `s`. Ví dụ `"(1, 3)"` thành `s = "(13)"`, và `"(2, 0.5)"` thành `s = "(205)"`.

Trả về danh sách các chuỗi biểu diễn **mọi khả năng** tọa độ gốc có thể là.

Quy ước số trong tọa độ gốc: **không có số 0 thừa**, nghĩa là không có các số như `"00"`, `"0.0"`, `"0.00"`, `"1.0"`, `"001"`, `"00.01"` (mọi số có thể viết bằng ít chữ số hơn). Ngoài ra **dấu chấm thập phân luôn có ít nhất một chữ số đứng trước**, nên không có `".1"`.

Kết quả có thể ở **thứ tự bất kỳ**. Mỗi tọa độ có đúng một khoảng trắng sau dấu phẩy.

**Ví dụ 1:** `s = "(123)"` -> `["(1, 2.3)", "(1, 23)", "(1.2, 3)", "(12, 3)"]`

**Ví dụ 2:** `s = "(0123)"` -> `["(0, 1.23)", "(0, 12.3)", "(0, 123)", "(0.1, 2.3)", "(0.1, 23)", "(0.12, 3)"]` (không cho phép `0.0`, `00`, `0001`, `00.01`).

**Ví dụ 3:** `s = "(00011)"` -> `["(0, 0.011)", "(0.001, 1)"]`

**Ràng buộc:**

- `4 <= s.length <= 12`
- `s[0] == '('` và `s[s.length - 1] == ')'`.
- Phần còn lại của `s` là các chữ số.

In [1]:
from typing import List
from decimal import Decimal
import re, random, time
from itertools import product

examples = [
    ("(123)", ["(1, 2.3)", "(1, 23)", "(1.2, 3)", "(12, 3)"]),
    ("(0123)", ["(0, 1.23)", "(0, 12.3)", "(0, 123)", "(0.1, 2.3)", "(0.1, 23)", "(0.12, 3)"]),
    ("(00011)", ["(0, 0.011)", "(0.001, 1)"]),
]
for s, expected in examples:
    print(f"s={s!r:10} số kết quả kỳ vọng = {len(expected)}")

s='(123)'    số kết quả kỳ vọng = 4
s='(0123)'   số kết quả kỳ vọng = 6
s='(00011)'  số kết quả kỳ vọng = 2


## 3. Kiến thức nền

### 3.1. Cắt chuỗi và bỏ dấu ngoặc

Chuỗi đầu vào dạng `"(" + chữ số + ")"`. Phần chữ số là `s[1:-1]`. Một chuỗi chữ số độ dài `m` có `m - 1` "khe" giữa các chữ số: mỗi khe là một chỗ có thể đặt **dấu phẩy** (ở bước tách hai vế) hoặc **dấu chấm** (trong một vế).

In [2]:
s = "(0123)"
digits = s[1:-1]
print(digits, "độ dài", len(digits), "-> số khe giữa các chữ số:", len(digits) - 1)
for k in range(1, len(digits)):
    print(f"khe {k}: {digits[:k]!r} | {digits[k:]!r}")

0123 độ dài 4 -> số khe giữa các chữ số: 3
khe 1: '0' | '123'
khe 2: '01' | '23'
khe 3: '012' | '3'


### 3.2. `startswith`, `endswith`

`t.startswith("0")` kiểm tra chuỗi bắt đầu bằng `"0"`; `t.endswith("0")` kiểm tra kết thúc bằng `"0"`. Hai phép này cho ta bắt "số 0 thừa" ở đầu phần nguyên và ở cuối phần thập phân.

In [3]:
for t in ["0", "01", "10", "100", "05", ""]:
    print(f"{t!r:6} startswith('0')={t.startswith('0')!s:5} endswith('0')={t.endswith('0')}")

'0'    startswith('0')=True  endswith('0')=True
'01'   startswith('0')=True  endswith('0')=False
'10'   startswith('0')=False endswith('0')=True
'100'  startswith('0')=False endswith('0')=True
'05'   startswith('0')=True  endswith('0')=False
''     startswith('0')=False endswith('0')=False


### 3.3. Luật "số hợp lệ"

Một số được viết `phần_nguyên` hoặc `phần_nguyên.phần_thập_phân`.

- **Phần nguyên** hợp lệ khi **là đúng `"0"`** hoặc **không bắt đầu bằng `"0"`** (cấm `00`, `01`, `007`).
- **Phần thập phân** (nếu có) hợp lệ khi **không kết thúc bằng `"0"`** (cấm `0.0`, `1.50`, `0.10`).

Ví dụ: `0.5` hợp lệ; `00.5` (phần nguyên `00`), `0.50` (phần thập phân `50`), `1.0` (phần thập phân `0`) đều không hợp lệ; `10` hợp lệ (số nguyên có chữ số 0 ở cuối là bình thường vì không có dấu chấm).

In [4]:
def valid_number(t: str) -> bool:
    if "." in t:
        left, right = t.split(".")
        return bool(right) and (left == "0" or not left.startswith("0")) and left != "" and not right.endswith("0")
    return t == "0" or (t != "" and not t.startswith("0"))

for t in ["0", "5", "10", "0.5", "12.34", "00", "01", "0.0", "1.0", "0.50", "00.5", ".5", "1."]:
    print(f"{t!r:8} -> {valid_number(t)}")

'0'      -> True
'5'      -> True
'10'     -> True
'0.5'    -> True
'12.34'  -> True
'00'     -> False
'01'     -> False
'0.0'    -> False
'1.0'    -> False
'0.50'   -> False
'00.5'   -> False
'.5'     -> False
'1.'     -> False


### 3.4. Tích Descartes bằng vòng lặp lồng nhau

Nếu có danh sách `A` các lựa chọn cho vế trái và `B` cho vế phải thì mọi ghép cặp là `[(a, b) for a in A for b in B]` (hoặc `itertools.product(A, B)`). Với ví dụ `"(123)"` tách `1 | 23` cho `A = ["1"]`, `B = ["2.3", "23"]` nên có 2 tọa độ.

In [5]:
A = ["1"]
B = ["2.3", "23"]
print([f"({a}, {b})" for a in A for b in B])
print(list(product(A, B)))

['(1, 2.3)', '(1, 23)']
[('1', '2.3'), ('1', '23')]


### 3.5. So sánh danh sách không phụ thuộc thứ tự

Đề cho phép trả về theo thứ tự bất kỳ, nên hai lời giải đúng có thể cho hai danh sách **cùng phần tử khác thứ tự**. Khi test, so sánh `sorted(a) == sorted(b)` (và kiểm tra **không có phần tử trùng**: `len(set(a)) == len(a)`).

In [6]:
a = ["(1, 23)", "(12, 3)"]
b = ["(12, 3)", "(1, 23)"]
print(a == b, sorted(a) == sorted(b), len(set(a)) == len(a))

False True True


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Với `"(123)"`, chữ số là `123`. Chọn dấu phẩy:

- `1 | 23`: vế trái `1` (chỉ `1`); vế phải `23` (có thể `23` hoặc `2.3`) -> `(1, 23)`, `(1, 2.3)`.
- `12 | 3`: vế trái `12` hoặc `1.2`; vế phải `3` -> `(12, 3)`, `(1.2, 3)`.

Tổng 4 tọa độ, khớp ví dụ 1.

Với `"(00011)"`, chữ số `00011`:

| tách | vế trái | số hợp lệ của vế trái | vế phải | số hợp lệ của vế phải |
|---|---|---|---|---|
| `0 | 0011` | `0` | `0` | `0011` | `0.011` |
| `00 | 011` | `00` | (không có) | | |
| `000 | 11` | `000` | (không có) | | |
| `0001 | 1` | `0001` | `0.001` | `1` | `1` |

Chỉ hai tách có cả hai vế hợp lệ: `(0, 0.011)` và `(0.001, 1)`. Rất nhiều tách bị loại vì vế trái toàn số 0 (`00`, `000`) hoặc dạng `0001` (có thể viết `0.001` nên **số nguyên `0001` bị cấm**, còn dạng thập phân `0.001` thì ổn).

Quy trình: **với mỗi cách tách đôi, tính danh sách số hợp lệ của từng vế rồi ghép cặp**.

In [7]:
def numbers_for(d: str) -> List[str]:
    """Mọi số hợp lệ tạo được từ chuỗi chữ số d (không phụ thuộc thứ tự)."""
    out = []
    for k in range(1, len(d) + 1):
        left, right = d[:k], d[k:]
        if (left == "0" or not left.startswith("0")) and not right.endswith("0"):
            out.append(left + ("." if right else "") + right)
    return out

for d in ["1", "23", "0011", "0001", "00", "100", "0", "05"]:
    print(f"{d!r:7} -> {numbers_for(d)}")

'1'     -> ['1']
'23'    -> ['2.3', '23']
'0011'  -> ['0.011']
'0001'  -> ['0.001']
'00'    -> []
'100'   -> ['100']
'0'     -> ['0']
'05'    -> ['0.5']


## 5. Cách 1 — Brute Force: liệt kê thô rồi lọc bằng định nghĩa

**Ý tưởng.** Không dùng bất kỳ luật nào của mục 3.3 để *sinh*. Với mỗi vị trí dấu phẩy và mỗi vế, **sinh mọi cách** đặt dấu chấm (không đặt, hoặc đặt ở một trong các khe), rồi **lọc** theo định nghĩa của đề: "số không thể viết bằng ít chữ số hơn". Ta kiểm tra điều đó bằng `Decimal`: số `t` hợp lệ khi **chuỗi chuẩn tắc** của giá trị số trùng với chính `t`.

**Giải thích từng đoạn.**

- `canonical(t)`: `format(Decimal(t).normalize(), "f") == t`. `normalize()` bỏ số 0 thừa; `"f"` in dạng thập phân (không dùng ký hiệu mũ). Ví dụ `"0.0"` chuẩn hóa thành `"0"` khác `"0.0"` nên bị loại.
- `raw_numbers(d)`: `d` (không chấm) và mọi `d[:k] + "." + d[k:]`.
- Ghép cặp qua mọi vị trí dấu phẩy.

Đây là mô phỏng trực tiếp định nghĩa nên cũng là **oracle**.

In [8]:
def canonical(t: str) -> bool:
    return format(Decimal(t).normalize(), "f") == t

def raw_numbers(d: str) -> List[str]:
    return [d] + [d[:k] + "." + d[k:] for k in range(1, len(d))]

def amb_brute(s: str) -> List[str]:
    digits = s[1:-1]
    res = []
    for i in range(1, len(digits)):
        for x in raw_numbers(digits[:i]):
            if not canonical(x):
                continue
            for y in raw_numbers(digits[i:]):
                if canonical(y):
                    res.append(f"({x}, {y})")
    return res

for s, expected in examples:
    got = amb_brute(s)
    print(s, "->", sorted(got) == sorted(expected), "| số kết quả", len(got))
    assert sorted(got) == sorted(expected)

(123) -> True | số kết quả 4
(0123) -> True | số kết quả 6
(00011) -> True | số kết quả 2


**Độ phức tạp và nút thắt.** Có `m - 1` vị trí phẩy (`m` là số chữ số, `m <= 10`), mỗi vế có tối đa `m` cách đặt chấm; tổng cỡ `m^3` cặp, và với mỗi cặp ta gọi `Decimal` (chậm). Với `m <= 10` vẫn nhỏ (vài trăm cặp), nhưng `Decimal` làm hằng số lớn và ta sinh nhiều chuỗi **chắc chắn sai** (như `"00"`). Ta có thể **không sinh** chúng ngay từ đầu.

In [9]:
for m in (2, 4, 6, 8, 10):
    digits = "0" * m                         # toàn số 0: gần như mọi ứng viên đều sai
    raw = sum(len(raw_numbers(digits[:i])) * len(raw_numbers(digits[i:])) for i in range(1, m))
    print(f"{m:2} chữ số: sinh thô {raw:4} cặp, hợp lệ {len(amb_brute('(' + digits + ')'))}")

 2 chữ số: sinh thô    1 cặp, hợp lệ 1
 4 chữ số: sinh thô   10 cặp, hợp lệ 0
 6 chữ số: sinh thô   35 cặp, hợp lệ 0
 8 chữ số: sinh thô   84 cặp, hợp lệ 0
10 chữ số: sinh thô  165 cặp, hợp lệ 0


## 6. Ý tưởng thuật toán / bước đột phá

**Độc lập hai vế + luật cục bộ.** Số hợp lệ của một vế chỉ phụ thuộc vào **chuỗi chữ số của vế đó**, không phụ thuộc vế kia. Vậy:

1. Với mỗi vị trí dấu phẩy `i`, tính `numbers(vế trái)` và `numbers(vế phải)` **riêng**.
2. Ghép cặp (tích Descartes).

Và `numbers(d)` có thể sinh **chỉ các số hợp lệ**, nhờ luật: tách `d = left + right` tại vị trí dấu chấm (`right` rỗng nghĩa là không có chấm): hợp lệ khi `left == "0"` hoặc `left` không bắt đầu bằng `0`, **và** `right` không kết thúc bằng `0`.

Ba cách triển khai khác nhau của `numbers(d)`:

| Cách | Ý tưởng |
|---|---|
| 2 | Vòng lặp vị trí chấm + luật `startswith` / `endswith` |
| 3 | **Phân tích trường hợp** theo ký tự đầu và cuối của `d`, xây trực tiếp (không vòng lặp kiểm tra) |
| 4 | Sinh mọi ứng viên rồi lọc bằng **biểu thức chính quy** `(0|[1-9]\d*)(\.\d*[1-9])?` |
| 5 | **Quay lui từng ký tự** trên toàn chuỗi (đặt/không đặt `.` hoặc `,` ở mỗi khe) có cắt tỉa sớm |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Vòng lặp vị trí chấm + luật đầu/cuối

**Giải thích từng đoạn.**

- `f(i, j)` trả mọi số hợp lệ từ `s[i:j]` (dùng trực tiếp chỉ số trên `s`, tránh cắt chuỗi thừa).
- Với `k` từ `1` tới `j - i`: `l = s[i : i + k]` là phần nguyên, `r = s[i + k : j]` là phần thập phân (có thể rỗng khi `k` chạm hết đoạn).
- Hợp lệ khi `(l == "0" or not l.startswith("0"))` và `not r.endswith("0")`; chuỗi số là `l + "." + r` nếu `r` không rỗng, ngược lại `l`.
- Vòng chính: vị trí dấu phẩy `i` từ `2` tới `n - 2` (chỉ số trong `s`, bao cả dấu ngoặc), ghép `f(1, i)` với `f(i, n - 1)`.

In [10]:
def amb_split(s: str) -> List[str]:
    def f(i: int, j: int) -> List[str]:
        res = []
        for k in range(1, j - i + 1):
            l, r = s[i:i + k], s[i + k:j]
            ok = (l == "0" or not l.startswith("0")) and not r.endswith("0")
            if ok:
                res.append(l + ("." if k < j - i else "") + r)
        return res

    n = len(s)
    return [f"({x}, {y})" for i in range(2, n - 1) for x in f(1, i) for y in f(i, n - 1)]

for s, expected in examples:
    assert sorted(amb_split(s)) == sorted(expected)
print("tách theo vị trí chấm OK")

tách theo vị trí chấm OK


**Code nộp LeetCode (Cách 2):**

In [11]:
class Solution:
    def ambiguousCoordinates(self, s: str) -> List[str]:
        def f(i, j):
            res = []
            for k in range(1, j - i + 1):
                l, r = s[i : i + k], s[i + k : j]
                ok = (l == "0" or not l.startswith("0")) and not r.endswith("0")
                if ok:
                    res.append(l + ("." if k < j - i else "") + r)
            return res

        n = len(s)
        return [
            f"({x}, {y})" for i in range(2, n - 1) for x in f(1, i) for y in f(i, n - 1)
        ]

sol_split = Solution().ambiguousCoordinates
for s, expected in examples:
    assert sorted(sol_split(s)) == sorted(expected)
print("Cách 2 OK")

Cách 2 OK


**Độ phức tạp (Cách 2):** gọi `m` là số chữ số (`m <= 10`). Mỗi `f` tốn `O(m)` lần thử cộng nối chuỗi `O(m)`; có `m - 1` vị trí phẩy và số cặp tối đa `O(m^2)`, mỗi kết quả dài `O(m)`: tổng `O(m^4)`, rất nhỏ với `m <= 10`. Bộ nhớ `O(m^3)` cho danh sách kết quả.

---

### 7.2. Cách 3 — Phân tích trường hợp, xây trực tiếp

**Ý tưởng khác.** Thay vì thử mọi vị trí chấm rồi kiểm tra, ta **suy ra ngay** các số hợp lệ từ ký tự đầu và cuối của `d`:

| `d` | số hợp lệ |
|---|---|
| dài > 1, bắt đầu `0`, kết thúc `0` (như `00`, `0010`) | **không có** (`0.xxx0` có số 0 thừa ở cuối; số nguyên `0xxx0` có số 0 đầu) |
| dài > 1, bắt đầu `0`, kết thúc khác `0` (như `0011`, `01`) | **chỉ** `0.` + phần sau (`0.011`, `0.1`): phần nguyên chỉ có thể là `0` |
| không bắt đầu bằng `0` (hoặc `d == "0"`), kết thúc `0` | **chỉ** `d` (số nguyên): mọi dấu chấm sẽ tạo phần thập phân kết thúc bằng `0` |
| không bắt đầu bằng `0`, kết thúc khác `0` | `d` **và** mọi `d[:k] + "." + d[k:]` với `1 <= k < len(d)` |

**Giải thích từng đoạn.** Hàm `numbers_case(d)` là bốn nhánh `if` ở trên; không có vòng lặp kiểm tra luật.

In [12]:
def numbers_case(d: str) -> List[str]:
    if len(d) > 1 and d[0] == "0":
        return [] if d[-1] == "0" else ["0." + d[1:]]
    if d[-1] == "0":
        return [d]
    return [d] + [d[:k] + "." + d[k:] for k in range(1, len(d))]

def amb_case(s: str) -> List[str]:
    digits = s[1:-1]
    return [f"({x}, {y})"
            for i in range(1, len(digits))
            for x in numbers_case(digits[:i])
            for y in numbers_case(digits[i:])]

# numbers_case khớp numbers_for trên mọi chuỗi chữ số 0/1/2 độ dài <= 7
for m in range(1, 8):
    for t in product("012", repeat=m):
        d = "".join(t)
        assert sorted(numbers_case(d)) == sorted(numbers_for(d)), d
for s, expected in examples:
    assert sorted(amb_case(s)) == sorted(expected)
print("phân tích trường hợp OK")

phân tích trường hợp OK


**Code nộp LeetCode (Cách 3):**

In [13]:
class Solution:
    def ambiguousCoordinates(self, s: str) -> List[str]:
        def numbers(d: str) -> List[str]:
            if len(d) > 1 and d[0] == "0":
                return [] if d[-1] == "0" else ["0." + d[1:]]
            if d[-1] == "0":
                return [d]
            return [d] + [d[:k] + "." + d[k:] for k in range(1, len(d))]

        digits = s[1:-1]
        return [f"({x}, {y})"
                for i in range(1, len(digits))
                for x in numbers(digits[:i])
                for y in numbers(digits[i:])]

sol_case = Solution().ambiguousCoordinates
for s, expected in examples:
    assert sorted(sol_case(s)) == sorted(expected)
print("Cách 3 OK")

Cách 3 OK


**Độ phức tạp (Cách 3):** cùng bậc `O(m^4)` nhưng hằng số nhỏ hơn (không kiểm tra luật cho từng ứng viên sai); bộ nhớ `O(m^3)`.

---

### 7.3. Cách 4 — Sinh mọi ứng viên rồi lọc bằng regex

**Ý tưởng khác.** Số hợp lệ khớp mẫu: **phần nguyên** `0` hoặc `[1-9]\d*`, tùy chọn **phần thập phân** `\.\d*[1-9]` (một dấu chấm, các chữ số, kết thúc bằng chữ số khác 0). Ta sinh mọi ứng viên (như Cách 1) nhưng lọc bằng `re.fullmatch(mẫu, ứng_viên)` thay vì `Decimal`.

**Giải thích từng đoạn.** `NUMBER = re.compile(r"(0|[1-9]\d*)(\.\d*[1-9])?")` biên dịch một lần; `fullmatch` đòi khớp toàn bộ chuỗi (nên `"00"`, `"0.0"`, `"1.50"` đều bị loại).

In [14]:
NUMBER = re.compile(r"(0|[1-9]\d*)(\.\d*[1-9])?")

def amb_regex(s: str) -> List[str]:
    digits = s[1:-1]
    res = []
    for i in range(1, len(digits)):
        xs = [x for x in raw_numbers(digits[:i]) if NUMBER.fullmatch(x)]
        ys = [y for y in raw_numbers(digits[i:]) if NUMBER.fullmatch(y)]
        res += [f"({x}, {y})" for x in xs for y in ys]
    return res

for s, expected in examples:
    assert sorted(amb_regex(s)) == sorted(expected)
print("regex OK")

regex OK


**Code nộp LeetCode (Cách 4):**

In [15]:
class Solution:
    def ambiguousCoordinates(self, s: str) -> List[str]:
        import re
        number = re.compile(r"(0|[1-9]\d*)(\.\d*[1-9])?")

        def candidates(d: str) -> List[str]:
            return [d] + [d[:k] + "." + d[k:] for k in range(1, len(d))]

        digits = s[1:-1]
        res = []
        for i in range(1, len(digits)):
            xs = [x for x in candidates(digits[:i]) if number.fullmatch(x)]
            ys = [y for y in candidates(digits[i:]) if number.fullmatch(y)]
            res += [f"({x}, {y})" for x in xs for y in ys]
        return res

sol_regex = Solution().ambiguousCoordinates
for s, expected in examples:
    assert sorted(sol_regex(s)) == sorted(expected)
print("Cách 4 OK")

Cách 4 OK


**Độ phức tạp (Cách 4):** `O(m^4)` (thêm chi phí khớp regex mỗi ứng viên, vẫn nhỏ); bộ nhớ `O(m^3)`.

---

### 7.4. Cách 5 — Quay lui từng ký tự có cắt tỉa

**Ý tưởng khác.** Duyệt các chữ số từ trái sang phải. Ở mỗi **khe** giữa hai chữ số có tối đa ba lựa chọn: **không đặt gì**, **đặt `.`** (kết thúc phần nguyên của số hiện tại) hoặc **đặt `,`** (kết thúc số trái, bắt đầu số phải). Trạng thái là `stage`: 0 = phần nguyên của số trái, 1 = phần thập phân của số trái, 2 = phần nguyên của số phải, 3 = phần thập phân của số phải.

**Giải thích từng đoạn.**

- Khi **kết thúc phần nguyên** (bằng `.`, bằng `,` hoặc hết chuỗi): phần nguyên phải là `0` hoặc không bắt đầu bằng `0`.
- Khi **kết thúc phần thập phân** (bằng `,` hoặc hết chuỗi): chữ số cuối khác `0`.
- **Cắt tỉa sớm:** nếu phần nguyên hiện là `"0"` và ta định thêm chữ số nữa **mà không đặt dấu chấm**, phần nguyên sẽ thành `0x` (không hợp lệ) nên bỏ nhánh đó ngay.
- Khi hết chữ số, chỉ chấp nhận `stage` 2 hoặc 3 (đã có dấu phẩy).

In [16]:
def amb_backtrack(s: str) -> List[str]:
    digits = s[1:-1]
    n = len(digits)
    res = []

    def int_ok(p: str) -> bool:
        return p == "0" or not p.startswith("0")

    def frac_ok(p: str) -> bool:
        return not p.endswith("0")

    def go(i, stage, ip, fp, left, text):
        # i: chữ số kế tiếp cần xử lý; ip/fp: phần nguyên/thập phân hiện tại; left: số trái đã hoàn tất; text: số hiện tại đang dựng
        if i == n:
            if stage == 2 and int_ok(ip):
                res.append(f"({left}, {text})")
            elif stage == 3 and int_ok(ip) and frac_ok(fp):
                res.append(f"({left}, {text})")
            return
        d = digits[i]
        # 1) không đặt dấu gì: nối chữ số vào phần hiện tại
        if stage in (0, 2):
            if ip != "0":                                   # cắt tỉa: "0" + chữ số nữa (không chấm) là sai
                go(i + 1, stage, ip + d, fp, left, text + d)
        else:
            go(i + 1, stage, ip, fp + d, left, text + d)
        # 2) đặt dấu chấm (chỉ khi đang ở phần nguyên và phần nguyên hợp lệ)
        if stage in (0, 2) and int_ok(ip):
            go(i + 1, stage + 1, ip, d, left, text + "." + d)
        # 3) đặt dấu phẩy (chỉ khi chưa có và số trái hợp lệ)
        if stage in (0, 1):
            ok = int_ok(ip) if stage == 0 else (int_ok(ip) and frac_ok(fp))
            if ok:
                go(i + 1, 2, d, "", text, d)

    go(1, 0, digits[0], "", "", digits[0])
    return res

for s, expected in examples:
    assert sorted(amb_backtrack(s)) == sorted(expected)
print("quay lui từng ký tự OK")

quay lui từng ký tự OK


**Code nộp LeetCode (Cách 5):**

In [17]:
class Solution:
    def ambiguousCoordinates(self, s: str) -> List[str]:
        digits = s[1:-1]
        n = len(digits)
        res = []

        def int_ok(p):
            return p == "0" or not p.startswith("0")

        def frac_ok(p):
            return not p.endswith("0")

        def go(i, stage, ip, fp, left, text):
            if i == n:
                if (stage == 2 and int_ok(ip)) or (stage == 3 and int_ok(ip) and frac_ok(fp)):
                    res.append(f"({left}, {text})")
                return
            d = digits[i]
            if stage in (0, 2):
                if ip != "0":
                    go(i + 1, stage, ip + d, fp, left, text + d)
            else:
                go(i + 1, stage, ip, fp + d, left, text + d)
            if stage in (0, 2) and int_ok(ip):
                go(i + 1, stage + 1, ip, d, left, text + "." + d)
            if stage in (0, 1):
                ok = int_ok(ip) if stage == 0 else (int_ok(ip) and frac_ok(fp))
                if ok:
                    go(i + 1, 2, d, "", text, d)

        go(1, 0, digits[0], "", "", digits[0])
        return res

sol_backtrack = Solution().ambiguousCoordinates
for s, expected in examples:
    assert sorted(sol_backtrack(s)) == sorted(expected)
print("Cách 5 OK")

Cách 5 OK


**Độ phức tạp (Cách 5):** số nhánh tối đa `3^(m-1)` nhưng cắt tỉa (không chấm sau `0`, kiểm tra hợp lệ khi đóng số) làm giảm mạnh; với `m <= 10` rất nhỏ. Bộ nhớ `O(m)` cho ngăn xếp đệ quy cộng danh sách kết quả.

## 8. TRACE từng bước

### 8.1. Cách 2 trên `s = "(123)"`

`s = "(123)"`, `n = 5`, vị trí phẩy `i` chạy `2, 3` (tách `s[1:i]` và `s[i:4]`):

| `i` | vế trái `s[1:i]` | `f(1, i)` | vế phải `s[i:4]` | `f(i, 4)` | tọa độ |
|---|---|---|---|---|---|
| 2 | `1` | `["1"]` | `23` | `["2.3", "23"]` | `(1, 2.3)`, `(1, 23)` |
| 3 | `12` | `["1.2", "12"]` | `3` | `["3"]` | `(1.2, 3)`, `(12, 3)` |

Hàm `trace_split` in từng bước.

In [18]:
def trace_split(s: str):
    n = len(s)
    print(f"s = {s!r}, n = {n}")

    def f(i, j, label):
        res = []
        print(f"    f({i}, {j}) cho {label} = {s[i:j]!r}:")
        for k in range(1, j - i + 1):
            l, r = s[i:i + k], s[i + k:j]
            ok_l = l == "0" or not l.startswith("0")
            ok_r = not r.endswith("0")
            tag = "ok" if (ok_l and ok_r) else ("phần nguyên sai" if not ok_l else "phần thập phân sai")
            print(f"      k={k}: phần nguyên {l!r}, thập phân {r!r} -> {tag}")
            if ok_l and ok_r:
                res.append(l + ("." if k < j - i else "") + r)
        print(f"    => {res}")
        return res

    out = []
    for i in range(2, n - 1):
        print(f"  dấu phẩy sau vị trí {i - 1}: trái = {s[1:i]!r}, phải = {s[i:n - 1]!r}")
        xs = f(1, i, "vế trái")
        ys = f(i, n - 1, "vế phải")
        for x in xs:
            for y in ys:
                out.append(f"({x}, {y})")
    print("kết quả:", out)
    return out

assert sorted(trace_split("(123)")) == sorted(examples[0][1])

s = '(123)', n = 5
  dấu phẩy sau vị trí 1: trái = '1', phải = '23'
    f(1, 2) cho vế trái = '1':
      k=1: phần nguyên '1', thập phân '' -> ok
    => ['1']
    f(2, 4) cho vế phải = '23':
      k=1: phần nguyên '2', thập phân '3' -> ok
      k=2: phần nguyên '23', thập phân '' -> ok
    => ['2.3', '23']
  dấu phẩy sau vị trí 2: trái = '12', phải = '3'
    f(1, 3) cho vế trái = '12':
      k=1: phần nguyên '1', thập phân '2' -> ok
      k=2: phần nguyên '12', thập phân '' -> ok
    => ['1.2', '12']
    f(3, 4) cho vế phải = '3':
      k=1: phần nguyên '3', thập phân '' -> ok
    => ['3']
kết quả: ['(1, 2.3)', '(1, 23)', '(1.2, 3)', '(12, 3)']


### 8.2. Vì sao `"(00011)"` chỉ có 2 kết quả

Chạy vết cho ví dụ 3 để thấy các tách bị loại ở đâu.

In [19]:
assert sorted(trace_split("(00011)")) == sorted(examples[2][1])

s = '(00011)', n = 7
  dấu phẩy sau vị trí 1: trái = '0', phải = '0011'
    f(1, 2) cho vế trái = '0':
      k=1: phần nguyên '0', thập phân '' -> ok
    => ['0']
    f(2, 6) cho vế phải = '0011':
      k=1: phần nguyên '0', thập phân '011' -> ok
      k=2: phần nguyên '00', thập phân '11' -> phần nguyên sai
      k=3: phần nguyên '001', thập phân '1' -> phần nguyên sai
      k=4: phần nguyên '0011', thập phân '' -> phần nguyên sai
    => ['0.011']
  dấu phẩy sau vị trí 2: trái = '00', phải = '011'
    f(1, 3) cho vế trái = '00':
      k=1: phần nguyên '0', thập phân '0' -> phần thập phân sai
      k=2: phần nguyên '00', thập phân '' -> phần nguyên sai
    => []
    f(3, 6) cho vế phải = '011':
      k=1: phần nguyên '0', thập phân '11' -> ok
      k=2: phần nguyên '01', thập phân '1' -> phần nguyên sai
      k=3: phần nguyên '011', thập phân '' -> phần nguyên sai
    => ['0.11']
  dấu phẩy sau vị trí 3: trái = '000', phải = '11'
    f(1, 4) cho vế trái = '000':
      k=1: phần nguyên 

### 8.3. Cách 3 (bốn trường hợp) trên một số chuỗi chữ số

Bảng chọn nhánh cho từng `d`:

In [20]:
def explain_case(d):
    if len(d) > 1 and d[0] == "0":
        why = "bắt đầu 0 và kết thúc 0 -> không có" if d[-1] == "0" else "bắt đầu 0 (dài > 1), cuối khác 0 -> chỉ 0.xxx"
    elif d[-1] == "0":
        why = "kết thúc 0 -> chỉ số nguyên d"
    else:
        why = "không bắt đầu 0, cuối khác 0 -> d và mọi chỗ chấm"
    print(f"{d!r:8} {why:48} => {numbers_case(d)}")

for d in ["12", "1", "0", "01", "011", "00", "010", "10", "100", "205"]:
    explain_case(d)

'12'     không bắt đầu 0, cuối khác 0 -> d và mọi chỗ chấm => ['12', '1.2']
'1'      không bắt đầu 0, cuối khác 0 -> d và mọi chỗ chấm => ['1']
'0'      kết thúc 0 -> chỉ số nguyên d                    => ['0']
'01'     bắt đầu 0 (dài > 1), cuối khác 0 -> chỉ 0.xxx    => ['0.1']
'011'    bắt đầu 0 (dài > 1), cuối khác 0 -> chỉ 0.xxx    => ['0.11']
'00'     bắt đầu 0 và kết thúc 0 -> không có              => []
'010'    bắt đầu 0 và kết thúc 0 -> không có              => []
'10'     kết thúc 0 -> chỉ số nguyên d                    => ['10']
'100'    kết thúc 0 -> chỉ số nguyên d                    => ['100']
'205'    không bắt đầu 0, cuối khác 0 -> d và mọi chỗ chấm => ['205', '2.05', '20.5']


### 8.4. Cách 5: cây quay lui cho `"(205)"`

Chữ số `205`: tách và đặt chấm. In ra các kết quả đạt được và so với Cách 2:

In [21]:
res = amb_backtrack("(205)")
print(sorted(res))
print("Cách 2 cho cùng input:", sorted(amb_split("(205)")))
assert sorted(res) == sorted(amb_split("(205)")) == ["(2, 0.5)", "(20, 5)"]

['(2, 0.5)', '(20, 5)']
Cách 2 cho cùng input: ['(2, 0.5)', '(20, 5)']


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (vòng lặp vị trí chấm + luật đầu/cuối): ngắn, rõ nghĩa và là lời giải chuẩn.

```python
class Solution:
    def ambiguousCoordinates(self, s: str) -> List[str]:
        def f(i, j):
            res = []
            for k in range(1, j - i + 1):
                l, r = s[i : i + k], s[i + k : j]
                ok = (l == "0" or not l.startswith("0")) and not r.endswith("0")
                if ok:
                    res.append(l + ("." if k < j - i else "") + r)
            return res

        n = len(s)
        return [
            f"({x}, {y})" for i in range(2, n - 1) for x in f(1, i) for y in f(i, n - 1)
        ]
```

In [22]:
class Solution:
    def ambiguousCoordinates(self, s: str) -> List[str]:
        def f(i, j):
            res = []
            for k in range(1, j - i + 1):
                l, r = s[i : i + k], s[i + k : j]
                ok = (l == "0" or not l.startswith("0")) and not r.endswith("0")
                if ok:
                    res.append(l + ("." if k < j - i else "") + r)
            return res

        n = len(s)
        return [
            f"({x}, {y})" for i in range(2, n - 1) for x in f(1, i) for y in f(i, n - 1)
        ]

sol = Solution()
for s, expected in examples:
    got = sol.ambiguousCoordinates(s)
    print(s, "->", sorted(got) == sorted(expected), len(got))
    assert sorted(got) == sorted(expected) and len(set(got)) == len(got)

(123) -> True 4
(0123) -> True 6
(00011) -> True 2


## 10. Vì sao đúng

**(a) Tách thành hai vế độc lập.** Tọa độ gốc `(x, y)` có đúng một dấu phẩy, hai số `x`, `y` không rỗng và các chữ số của `x` theo sau là các chữ số của `y` tạo lại đúng chuỗi chữ số của `s`. Vậy mỗi tọa độ tương ứng **một cách chọn** (vị trí phẩy, cách đặt chấm của `x`, cách đặt chấm của `y`), và hai cách đặt chấm độc lập. Cách chọn khác nhau cho chuỗi khác nhau (chuỗi `"(x, y)"` xác định vị trí phẩy và chấm), nên **không có phần tử trùng**.

**(b) Luật số hợp lệ.** Một số là hợp lệ nếu không thể viết bằng ít chữ số hơn và không bắt đầu bằng `.`:

- Phần nguyên `L` với `len(L) >= 2` bắt đầu bằng `0` luôn bỏ được số 0 đầu -> không hợp lệ. `L = "0"` hợp lệ.
- Phần thập phân `R` (có ít nhất một chữ số) kết thúc bằng `0` luôn bỏ được số 0 cuối -> không hợp lệ.
- Không có số 0 thừa nào khác: nếu `L` hợp lệ và `R` không kết thúc bằng 0 thì số có biểu diễn ngắn nhất (giá trị xác định duy nhất cho dạng chuẩn tắc).
Đây chính là điều kiện của Cách 2, và được Cách 1 (`Decimal`) và Cách 4 (regex) kiểm lại bằng hai định nghĩa khác.

**(c) Cách 3.** Chỉ là bảng suy ra từ (b): bắt đầu `0` và dài > 1 -> phần nguyên duy nhất khả dĩ là `"0"`, nên mọi số đều dạng `0.` + phần còn lại (cần phần còn lại không kết thúc `0`); kết thúc `0` -> mọi số có phần thập phân sẽ kết thúc `0` (vì chữ số cuối thuộc phần thập phân khi có chấm) nên chỉ còn số nguyên.

**(d) Cách 5.** Quay lui duyệt mọi cách đặt `.`/`,` vào các khe và chỉ chấp nhận khi mỗi số kết thúc thỏa luật (b); cắt tỉa `ip == "0"` chỉ loại các nhánh chắc chắn sai (phần nguyên `0x`).

**Kiểm chứng bằng máy:** so mọi cách (và `Solution`) với oracle (Cách 1, định nghĩa qua `Decimal`) trên **mọi** chuỗi chữ số trên bảng chữ cái `{0, 1}` độ dài 2..10, trên `{0, 1, 2}` độ dài 2..7 và trên `{0..9}` độ dài 2..3, rồi trên các chuỗi ngẫu nhiên có seed với đủ 10 chữ số (ở mục 12).

In [23]:
solutions = {
    "brute": amb_brute,
    "split": amb_split,
    "case": amb_case,
    "regex": amb_regex,
    "backtrack": amb_backtrack,
    "sol_split": sol_split,
    "sol_case": sol_case,
    "sol_regex": sol_regex,
    "sol_backtrack": sol_backtrack,
    "final": Solution().ambiguousCoordinates,
}

def check_all(s):
    ref = sorted(amb_brute(s))
    assert len(set(ref)) == len(ref), s
    for name, fn in solutions.items():
        got = fn(s)
        assert len(set(got)) == len(got), (name, s, "có phần tử trùng")
        assert sorted(got) == ref, (name, s, sorted(got), ref)
    return len(ref)

total, nonempty = 0, 0
for alphabet, max_len in [("01", 10), ("012", 7), ("0123456789", 3)]:
    for m in range(2, max_len + 1):
        for t in product(alphabet, repeat=m):
            r = check_all("(" + "".join(t) + ")")
            total += 1
            nonempty += r > 0
print(f"so khớp oracle trên {total} chuỗi ({nonempty} chuỗi có kết quả, {total - nonempty} chuỗi không có kết quả nào) x {len(solutions)} cài đặt: OK")

so khớp oracle trên 6420 chuỗi (6344 chuỗi có kết quả, 76 chuỗi không có kết quả nào) x 10 cài đặt: OK


## 11. Time / Space Complexity

Gọi `m = len(s) - 2` là số chữ số (`2 <= m <= 10`).

| Cách | Time | Space |
|---|---|---|
| 1. Thô + `Decimal` | `O(m^4)` (hằng số lớn do `Decimal`) | `O(m^3)` |
| 2. Vòng lặp chấm + luật đầu/cuối | `O(m^4)` | `O(m^3)` |
| 3. Phân tích trường hợp | `O(m^4)` (ít ứng viên thừa) | `O(m^3)` |
| 4. Thô + regex | `O(m^4)` | `O(m^3)` |
| 5. Quay lui có cắt tỉa | `O(số kết quả * m)` thực tế | `O(m)` ngăn xếp + kết quả |

Số kết quả tối đa với `m = 10` chữ số (chữ số khác 0, để mọi tọa độ hợp lệ) là cỡ vài trăm; đo thử:

In [24]:
rng = random.Random(816)
tests = ["(" + "".join(rng.choice("123456789") for _ in range(10)) + ")" for _ in range(200)]
tests += ["(" + "".join(rng.choice("0123456789") for _ in range(rng.randint(2, 10))) + ")" for _ in range(800)]
ref = None
for name in ["brute", "split", "case", "regex", "backtrack"]:
    t0 = time.perf_counter()
    res = [sorted(solutions[name](s)) for s in tests]
    dt = (time.perf_counter() - t0) * 1000
    if ref is None:
        ref = res
    assert res == ref, name
    print(f"{name:10} {len(tests)} chuỗi: {dt:8.1f} ms")
mx = max(len(r) for r in ref)
print("số kết quả lớn nhất trong các test:", mx, "(chuỗi", tests[[len(r) for r in ref].index(mx)], ")")

brute      1000 chuỗi:     88.7 ms
split      1000 chuỗi:     39.6 ms
case       1000 chuỗi:     33.5 ms


regex      1000 chuỗi:     39.3 ms
backtrack  1000 chuỗi:     75.3 ms
số kết quả lớn nhất trong các test: 165 (chuỗi (3278215645) )


## 12. Bộ test

Gồm: ví dụ của đề (so sánh sau khi sắp xếp), edge case, phản ví dụ cho các cách sai, đối chiếu mọi cách với oracle (định nghĩa qua `Decimal`), và test ngẫu nhiên có seed (đúng ràng buộc `4 <= len(s) <= 12`).

In [25]:
# 12.1 ví dụ đề (thứ tự kết quả không quan trọng: so sánh sau khi sắp xếp)
for s, expected in examples:
    for name, fn in solutions.items():
        got = fn(s)
        assert sorted(got) == sorted(expected) and len(set(got)) == len(got), (name, s)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, đã suy luận từng ca rồi đối chiếu oracle)
edge = {
    "(00)": ["(0, 0)"],
    "(11)": ["(1, 1)"],
    "(10)": ["(1, 0)"],
    "(01)": ["(0, 1)"],
    "(100)": ["(10, 0)"],
    "(0000)": [],
    "(000)": [],
    "(001)": ["(0, 0.1)"],
    "(010)": ["(0, 10)", "(0.1, 0)"],
    "(123)": ["(1, 2.3)", "(1, 23)", "(1.2, 3)", "(12, 3)"],
    "(205)": ["(2, 0.5)", "(20, 5)"],
}
for s, expected in edge.items():
    assert sorted(amb_brute(s)) == sorted(expected), ("oracle khác kỳ vọng", s, sorted(amb_brute(s)))
    for name, fn in solutions.items():
        got = fn(s)
        assert sorted(got) == sorted(expected) and len(set(got)) == len(got), (name, s, got)
# chuỗi dài nhất theo ràng buộc: 10 chữ số
for s in ["(1111111111)", "(1234567890)", "(0000000000)", "(1000000001)"]:
    check_all(s)
print("12.2 edge case: OK", len(edge) + 4, "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 15 ca


In [26]:
# 12.3 phản ví dụ: các cách sai
def numbers_wrong(d, leading=True, trailing=True):
    out = []
    for k in range(1, len(d) + 1):
        l, r = d[:k], d[k:]
        ok = True
        if leading and not (l == "0" or not l.startswith("0")):
            ok = False
        if trailing and r.endswith("0"):
            ok = False
        if ok:
            out.append(l + ("." if r else "") + r)
    return out

def amb_no_trailing_rule(s):           # bỏ luật số 0 ở cuối phần thập phân
    d = s[1:-1]
    return [f"({x}, {y})" for i in range(1, len(d)) for x in numbers_wrong(d[:i], True, False) for y in numbers_wrong(d[i:], True, False)]

def amb_no_leading_rule(s):            # bỏ luật số 0 ở đầu phần nguyên
    d = s[1:-1]
    return [f"({x}, {y})" for i in range(1, len(d)) for x in numbers_wrong(d[:i], False, True) for y in numbers_wrong(d[i:], False, True)]

def amb_comma_range(s):                # cho phép dấu phẩy ở đầu/cuối (vế rỗng)
    d = s[1:-1]
    return [f"({x}, {y})" for i in range(0, len(d) + 1)
            for x in (numbers_case(d[:i]) if d[:i] else [""]) for y in (numbers_case(d[i:]) if d[i:] else [""])]

def amb_no_space(s):                   # quên khoảng trắng sau dấu phẩy
    return [t.replace(", ", ",") for t in amb_case(s)]

def amb_no_integer_form(s):            # quên trường hợp không có dấu chấm
    d = s[1:-1]
    def nums(x):
        return [x[:k] + "." + x[k:] for k in range(1, len(x)) if (x[:k] == "0" or not x[:k].startswith("0")) and not x[k:].endswith("0")]
    return [f"({a}, {b})" for i in range(1, len(d)) for a in nums(d[:i]) for b in nums(d[i:])]

wrongs = [
    ("bỏ luật số 0 cuối", amb_no_trailing_rule, "(100)"),
    ("bỏ luật số 0 đầu", amb_no_leading_rule, "(0011)"),
    ("dấu phẩy ở đầu/cuối", amb_comma_range, "(12)"),
    ("quên khoảng trắng", amb_no_space, "(12)"),
    ("quên số nguyên (không chấm)", amb_no_integer_form, "(123)"),
]
for name, fn, s in wrongs:
    truth = sorted(amb_brute(s))
    got = sorted(fn(s))
    print(f"{name:30} s={s}: sai = {got[:4]}{'...' if len(got) > 4 else ''} ({len(got)} kq), đúng = {truth}")
    assert got != truth, name
print("12.3 phản ví dụ: OK (các cách sai đều bị lộ)")

# 12.4 cẩn thận thứ tự: hai danh sách cùng phần tử khác thứ tự phải bằng nhau sau khi sắp xếp
a = amb_split("(123)")
b = list(reversed(a))
print("so sánh trực tiếp:", a == b, "| sau khi sắp xếp:", sorted(a) == sorted(b))
assert a != b and sorted(a) == sorted(b)

bỏ luật số 0 cuối              s=(100): sai = ['(1, 0.0)', '(1.0, 0)', '(10, 0)'] (3 kq), đúng = ['(10, 0)']
bỏ luật số 0 đầu               s=(0011): sai = ['(0, 0.11)', '(0, 01.1)', '(0, 011)', '(0.01, 1)']... (8 kq), đúng = ['(0, 0.11)', '(0.01, 1)']
dấu phẩy ở đầu/cuối            s=(12): sai = ['(, 1.2)', '(, 12)', '(1, 2)', '(1.2, )']... (5 kq), đúng = ['(1, 2)']
quên khoảng trắng              s=(12): sai = ['(1,2)'] (1 kq), đúng = ['(1, 2)']
quên số nguyên (không chấm)    s=(123): sai = [] (0 kq), đúng = ['(1, 2.3)', '(1, 23)', '(1.2, 3)', '(12, 3)']
12.3 phản ví dụ: OK (các cách sai đều bị lộ)
so sánh trực tiếp: False | sau khi sắp xếp: True


In [27]:
# 12.5 ngẫu nhiên có seed, đúng ràng buộc 4 <= len(s) <= 12 (2 <= số chữ số <= 10)
rng = random.Random(2816)
n_nonempty = 0
for _ in range(800):
    m = rng.randint(2, 10)
    alphabet = rng.choice(["01", "012", "0123456789", "123456789", "05"])
    s = "(" + "".join(rng.choice(alphabet) for _ in range(m)) + ")"
    n_nonempty += check_all(s) > 0
print(f"12.5 ngẫu nhiên: OK ({n_nonempty}/800 chuỗi có ít nhất một kết quả)")

# 12.6 chuỗi mà MỌI chữ số khác 0: số kết quả có công thức đóng. Với m chữ số khác 0, vế trái dài a có a số (a nguyên + a-1 vị trí chấm), vế phải dài b có b số
for m in range(2, 11):
    s = "(" + "123456789"[:min(m, 9)] + ("1" * (m - 9) if m > 9 else "") + ")"
    expected = sum(i * (m - i) for i in range(1, m))            # (số cách vế trái) * (số cách vế phải), mỗi vế dài L có L cách
    assert len(Solution().ambiguousCoordinates(s)) == expected, (m, len(Solution().ambiguousCoordinates(s)), expected)
print("12.6 công thức đóng cho chuỗi toàn chữ số khác 0 (sum i*(m-i)): OK")

12.5 ngẫu nhiên: OK (789/800 chuỗi có ít nhất một kết quả)
12.6 công thức đóng cho chuỗi toàn chữ số khác 0 (sum i*(m-i)): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bỏ luật số 0 ở cuối phần thập phân

Quên rằng `1.0`, `0.10`, `0.0` là số có thể viết ngắn hơn.

In [28]:
s = "(100)"
print("sai :", sorted(amb_no_trailing_rule(s)))
print("đúng:", sorted(Solution().ambiguousCoordinates(s)))
assert sorted(amb_no_trailing_rule(s)) != sorted(Solution().ambiguousCoordinates(s))

sai : ['(1, 0.0)', '(1.0, 0)', '(10, 0)']
đúng: ['(10, 0)']


Vì sao sai: `"(1, 0.0)"` hay `"(1.0, 0)"` có số 0 thừa. **Đúng:** loại phần thập phân kết thúc bằng `0`.

### Lỗi 2 — Bỏ luật số 0 ở đầu phần nguyên

Quên rằng `01`, `00`, `0011` không phải số hợp lệ ở dạng số nguyên.

In [29]:
s = "(0011)"
print("sai :", sorted(amb_no_leading_rule(s)))
print("đúng:", sorted(Solution().ambiguousCoordinates(s)))
assert sorted(amb_no_leading_rule(s)) != sorted(Solution().ambiguousCoordinates(s))

sai : ['(0, 0.11)', '(0, 01.1)', '(0, 011)', '(0.01, 1)', '(00, 1.1)', '(00, 11)', '(00.1, 1)', '(001, 1)']
đúng: ['(0, 0.11)', '(0.01, 1)']


Vì sao sai: `"0011"` có thể viết `"0.011"`; dạng `"0011"` bị cấm. **Đúng:** phần nguyên là `"0"` hoặc không bắt đầu bằng `"0"`.

### Lỗi 3 — Cho phép dấu phẩy ở đầu hoặc cuối

Mỗi vế phải có ít nhất một chữ số; vị trí phẩy chỉ ở `m - 1` khe giữa các chữ số.

In [30]:
s = "(12)"
print("sai :", sorted(amb_comma_range(s)))
print("đúng:", sorted(Solution().ambiguousCoordinates(s)))
assert sorted(amb_comma_range(s)) != sorted(Solution().ambiguousCoordinates(s))

sai : ['(, 1.2)', '(, 12)', '(1, 2)', '(1.2, )', '(12, )']
đúng: ['(1, 2)']


Vì sao sai: sinh ra `"(, 12)"` hoặc `"(12, )"`. **Đúng:** `for i in range(1, len(digits))` (hoặc `range(2, n - 1)` theo chỉ số trong `s`).

### Lỗi 4 — Quên khoảng trắng sau dấu phẩy

Đề yêu cầu đúng một khoảng trắng: `"(1, 23)"`, không phải `"(1,23)"`.

In [31]:
s = "(12)"
print("sai :", amb_no_space(s))
print("đúng:", Solution().ambiguousCoordinates(s))
assert amb_no_space(s) == ["(1,2)"] and Solution().ambiguousCoordinates(s) == ["(1, 2)"]

sai : ['(1,2)']
đúng: ['(1, 2)']


Vì sao sai: chuỗi kết quả không khớp định dạng nên bị chấm sai dù đúng nội dung. **Đúng:** `f"({x}, {y})"`.

### Lỗi 5 — Quên trường hợp không có dấu chấm

Mỗi vế có thể là **số nguyên** (không chấm) hoặc có chấm.

In [32]:
s = "(123)"
print("sai :", sorted(amb_no_integer_form(s)))
print("đúng:", sorted(Solution().ambiguousCoordinates(s)))
assert sorted(amb_no_integer_form(s)) != sorted(Solution().ambiguousCoordinates(s))

sai : []
đúng: ['(1, 2.3)', '(1, 23)', '(1.2, 3)', '(12, 3)']


Vì sao sai: các tọa độ như `"(1, 23)"` và `"(12, 3)"` không có chấm nào nên bị bỏ sót. **Đúng:** vòng `k` đi tới `j - i` (tức `right` rỗng, không chấm).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "khôi phục", "mọi khả năng", chuỗi ngắn (`<= 12`) -> **liệt kê** (enumeration) cả không gian, không cần thuật toán thông minh.
- **Tách thành phần độc lập** (hai vế) rồi **ghép cặp** bằng tích Descartes: đơn giản hơn nhiều so với duyệt chung.
- **Quy tắc hợp lệ cục bộ** (đầu `0`, cuối `0`) quyết định bài; nên viết hàm `valid` riêng và test độc lập.
- Khi thứ tự kết quả không quan trọng: **sắp xếp rồi so sánh**, và kiểm tra **không trùng**.
- Khi nghi ngờ luật, dùng **oracle định nghĩa** (như `Decimal` chuẩn tắc) để kiểm lại.
- Cùng họ: Restore IP Addresses (93), Generate Parentheses (22), Letter Combinations of a Phone Number (17), Additive Number (306).
- Test nên có: toàn `0`, chữ số kết thúc bằng `0`, chuỗi `"(00)"`, chuỗi dài 10 chữ số.

## 15. Biến thể

1. **Chỉ đếm** số khả năng (không liệt kê): `sum over i of count(trái) * count(phải)`, với `count(d)` suy ra từ bảng bốn trường hợp (Cách 3).
2. **Restore IP Addresses (93):** chia chuỗi chữ số thành 4 đoạn, mỗi đoạn `0..255` không có số 0 đầu (trừ `"0"`).
3. **Cho phép dấu âm** hoặc số mũ: thêm luật.
4. Bài liên quan: Additive Number (306), Different Ways to Add Parentheses (241).

Dưới đây là code cho biến thể 1 (đếm bằng công thức) và 2 (Restore IP), kiểm bằng oracle.

In [33]:
def count_numbers(d: str) -> int:
    if len(d) > 1 and d[0] == "0":
        return 0 if d[-1] == "0" else 1
    if d[-1] == "0":
        return 1
    return len(d)                                      # d và (len - 1) vị trí chấm

def count_coordinates(s: str) -> int:
    d = s[1:-1]
    return sum(count_numbers(d[:i]) * count_numbers(d[i:]) for i in range(1, len(d)))

rng = random.Random(15)
for _ in range(500):
    m = rng.randint(2, 10)
    s = "(" + "".join(rng.choice("0123") for _ in range(m)) + ")"
    assert count_coordinates(s) == len(amb_brute(s)), s
print("đếm bằng công thức khớp liệt kê: OK; ví dụ", count_coordinates("(123)"), count_coordinates("(00011)"))

def restore_ip(s: str) -> List[str]:
    def ok(p):
        return 0 < len(p) <= 3 and (p == "0" or not p.startswith("0")) and int(p) <= 255
    res = []
    n = len(s)
    for a in range(1, 4):
        for b in range(a + 1, a + 4):
            for c in range(b + 1, b + 4):
                if c < n and n - c <= 3:
                    parts = [s[:a], s[a:b], s[b:c], s[c:]]
                    if all(ok(p) for p in parts):
                        res.append(".".join(parts))
    return res

def restore_ip_oracle(s):
    from itertools import combinations
    out = []
    for cuts in combinations(range(1, len(s)), 3):
        parts = [s[:cuts[0]], s[cuts[0]:cuts[1]], s[cuts[1]:cuts[2]], s[cuts[2]:]]
        if all(str(int(p)) == p and 0 <= int(p) <= 255 for p in parts):
            out.append(".".join(parts))
    return out

assert sorted(restore_ip("25525511135")) == ["255.255.11.135", "255.255.111.35"]
assert restore_ip("0000") == ["0.0.0.0"] and sorted(restore_ip("101023")) == sorted(restore_ip_oracle("101023"))
for _ in range(300):
    s = "".join(random.Random(_ * 31 + i).choice("0123") for i in range(random.Random(_).randint(4, 12)))
    assert sorted(restore_ip(s)) == sorted(restore_ip_oracle(s)), s
print("Biến thể: OK")

đếm bằng công thức khớp liệt kê: OK; ví dụ 4 2
Biến thể: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thô + `Decimal` | Sinh mọi ứng viên, lọc bằng định nghĩa | `O(m^4)` (hằng số lớn) | `O(m^3)` | Oracle |
| 2. Vòng lặp chấm + luật | `startswith` / `endswith` | `O(m^4)` | `O(m^3)` | **Cách nộp chính** |
| 3. Phân tích trường hợp | Bốn nhánh theo đầu/cuối | `O(m^4)` nhỏ | `O(m^3)` | Nhanh nhất; thể hiện hiểu luật |
| 4. Thô + regex | `fullmatch` mẫu số hợp lệ | `O(m^4)` | `O(m^3)` | Muốn một biểu thức duy nhất cho định nghĩa |
| 5. Quay lui từng ký tự | Khe: không / chấm / phẩy, cắt tỉa | `O(kết quả * m)` | `O(m)` | Mở rộng sang bài có nhiều dấu hơn (IP, biểu thức) |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output (đã sắp xếp) của ba dòng sau, rồi chạy để kiểm tra:

```python
print(sorted(numbers_for("12")))
print(sorted(numbers_for("05")))
print(sorted(numbers_for("100")))
```

In [34]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `s = "(205)"`: liệt kê (a) các cách tách `trái | phải` của chuỗi chữ số, (b) số hợp lệ của từng vế, (c) danh sách tọa độ cuối cùng (đã sắp xếp).

In [35]:
# Bài 2: điền kết quả tính tay
hand_splits = None       # ví dụ [("2", "05"), ...]
hand_numbers = None      # dict: chuỗi chữ số -> list số hợp lệ
hand_answer = None       # list tọa độ, đã sắp xếp

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm sinh số hợp lệ.

In [36]:
def exercise_3_numbers(d: str) -> List[str]:
    res = []
    for k in range(1, len(d) + 1):
        left, right = d[:k], d[k:]
        # TODO 1: kiểm tra phần nguyên hợp lệ: left == "0" hoặc left không bắt đầu bằng "0"
        # TODO 2: kiểm tra phần thập phân hợp lệ: right không kết thúc bằng "0"
        # TODO 3: nếu cả hai hợp lệ thì thêm left (+ "." + right nếu right không rỗng) vào res
        pass
    return None


# assert sorted(exercise_3_numbers("12")) == ["1.2", "12"]
# assert exercise_3_numbers("05") == ["0.5"]
# assert exercise_3_numbers("100") == ["100"]
# assert exercise_3_numbers("00") == []

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_cases(s)` trả danh sách tọa độ dựa trên bốn trường hợp theo ký tự đầu và cuối của chuỗi chữ số.

In [37]:
def exercise_4_cases(s: str) -> List[str]:
    # TODO
    pass


# assert sorted(exercise_4_cases("(123)")) == ["(1, 2.3)", "(1, 23)", "(1.2, 3)", "(12, 3)"]
# assert sorted(exercise_4_cases("(00011)")) == ["(0, 0.011)", "(0.001, 1)"]
# assert exercise_4_cases("(0000)") == []

### Bài 5 — Mini challenge: Restore IP Addresses (khó)

Cho chuỗi chữ số `s` (độ dài 4..12). Trả về mọi địa chỉ IPv4 hợp lệ tạo được bằng cách chèn đúng 3 dấu chấm: bốn phần, mỗi phần là số `0..255` **không có số 0 đầu** (trừ chính `"0"`). Thứ tự kết quả bất kỳ.

```python
exercise_5_restore_ip("25525511135")  -> ["255.255.11.135", "255.255.111.35"]
exercise_5_restore_ip("0000")         -> ["0.0.0.0"]
exercise_5_restore_ip("101023")       -> 5 địa chỉ
```
Gợi ý: cùng ý "độc lập từng đoạn + luật hợp lệ cục bộ" như bài này; thử 3 vị trí cắt.

In [38]:
def exercise_5_restore_ip(s: str) -> List[str]:
    # TODO
    pass


# assert sorted(exercise_5_restore_ip("25525511135")) == ["255.255.11.135", "255.255.111.35"]
# assert exercise_5_restore_ip("0000") == ["0.0.0.0"]
# assert len(exercise_5_restore_ip("101023")) == 5

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `"12"` có thể giữ nguyên hoặc đặt chấm giữa hai chữ số.
- Mức 2: `"05"`: phần nguyên không được là `05`; `"100"`: phần thập phân không được kết thúc bằng 0.
- Mức 3: `['1.2', '12']`, `['0.5']`, `['100']`.

**Bài 2**
- Mức 1: chữ số là `205`; hai cách tách: `2 | 05` và `20 | 5`.
- Mức 2: `2` -> `["2"]`; `05` -> `["0.5"]`; `20` -> `["20"]`; `5` -> `["5"]`.
- Mức 3: `["(2, 0.5)", "(20, 5)"]`.

**Bài 3**
- Mức 1: `ok_left = left == "0" or not left.startswith("0")`.
- Mức 2: `ok_right = not right.endswith("0")`.
- Mức 3: `if ok_left and ok_right: res.append(left + ("." + right if right else ""))`; `return res`.

**Bài 4**
- Mức 1: viết hàm phụ `numbers(d)` với bốn nhánh.
- Mức 2: nhánh 1: `len(d) > 1 and d[0] == "0"`: không có nếu `d[-1] == "0"`, ngược lại `["0." + d[1:]]`.
- Mức 3: nhánh 2: `d[-1] == "0"` -> `[d]`; nhánh 3: `[d] + [d[:k] + "." + d[k:] ...]`; ghép cặp qua mọi vị trí phẩy.

**Bài 5**
- Mức 1: ba vòng lặp cho ba vị trí cắt (độ dài mỗi đoạn 1..3).
- Mức 2: hợp lệ khi `p == "0" or not p.startswith("0")` và `int(p) <= 255`.
- Mức 3: ghép `".".join(parts)`; nhớ điều kiện tổng độ dài bằng `len(s)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [39]:
print(sorted(numbers_for("12")))
print(sorted(numbers_for("05")))
print(sorted(numbers_for("100")))

assert sorted(numbers_for("12")) == ["1.2", "12"]
assert sorted(numbers_for("05")) == ["0.5"]
assert sorted(numbers_for("100")) == ["100"]

['1.2', '12']
['0.5']
['100']


### Đáp án Bài 2

Tách `2 | 05` và `20 | 5`; số hợp lệ: `2 -> ["2"]`, `05 -> ["0.5"]`, `20 -> ["20"]`, `5 -> ["5"]`; kết quả `["(2, 0.5)", "(20, 5)"]`.

In [40]:
hand_splits_solution = [("2", "05"), ("20", "5")]
hand_numbers_solution = {"2": ["2"], "05": ["0.5"], "20": ["20"], "5": ["5"]}
hand_answer_solution = ["(2, 0.5)", "(20, 5)"]
digits = "205"
assert [(digits[:i], digits[i:]) for i in range(1, len(digits))] == hand_splits_solution
assert {d: numbers_for(d) for d in hand_numbers_solution} == hand_numbers_solution
assert sorted(Solution().ambiguousCoordinates("(205)")) == hand_answer_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [41]:
def exercise_3_numbers_solution(d: str) -> List[str]:
    res = []
    for k in range(1, len(d) + 1):
        left, right = d[:k], d[k:]
        ok_left = left == "0" or not left.startswith("0")
        ok_right = not right.endswith("0")
        if ok_left and ok_right:
            res.append(left + ("." + right if right else ""))
    return res

assert sorted(exercise_3_numbers_solution("12")) == ["1.2", "12"]
assert exercise_3_numbers_solution("05") == ["0.5"]
assert exercise_3_numbers_solution("100") == ["100"]
assert exercise_3_numbers_solution("00") == []
for m in range(1, 8):
    for t in product("012", repeat=m):
        d = "".join(t)
        assert sorted(exercise_3_numbers_solution(d)) == sorted(numbers_for(d)), d
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [42]:
def exercise_4_cases_solution(s: str) -> List[str]:
    def numbers(d):
        if len(d) > 1 and d[0] == "0":
            return [] if d[-1] == "0" else ["0." + d[1:]]
        if d[-1] == "0":
            return [d]
        return [d] + [d[:k] + "." + d[k:] for k in range(1, len(d))]

    d = s[1:-1]
    return [f"({x}, {y})" for i in range(1, len(d)) for x in numbers(d[:i]) for y in numbers(d[i:])]

assert sorted(exercise_4_cases_solution("(123)")) == ["(1, 2.3)", "(1, 23)", "(1.2, 3)", "(12, 3)"]
assert sorted(exercise_4_cases_solution("(00011)")) == ["(0, 0.011)", "(0.001, 1)"]
assert exercise_4_cases_solution("(0000)") == []
rng = random.Random(4)
for _ in range(300):
    s = "(" + "".join(rng.choice("0123") for _ in range(rng.randint(2, 10))) + ")"
    assert sorted(exercise_4_cases_solution(s)) == sorted(amb_brute(s)), s
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [43]:
def exercise_5_restore_ip_solution(s: str) -> List[str]:
    def ok(p):
        return 0 < len(p) <= 3 and (p == "0" or not p.startswith("0")) and int(p) <= 255
    res = []
    n = len(s)
    for a in range(1, 4):
        for b in range(1, 4):
            for c in range(1, 4):
                d = n - a - b - c
                if 1 <= d <= 3:
                    parts = [s[:a], s[a:a + b], s[a + b:a + b + c], s[a + b + c:]]
                    if all(ok(p) for p in parts):
                        res.append(".".join(parts))
    return res

assert sorted(exercise_5_restore_ip_solution("25525511135")) == ["255.255.11.135", "255.255.111.35"]
assert exercise_5_restore_ip_solution("0000") == ["0.0.0.0"]
assert len(exercise_5_restore_ip_solution("101023")) == 5
rng = random.Random(5)
for _ in range(300):
    s = "".join(rng.choice("0123456789") for _ in range(rng.randint(4, 12)))
    got = exercise_5_restore_ip_solution(s)
    assert sorted(got) == sorted(restore_ip_oracle(s)) and len(set(got)) == len(got), s
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (vòng lặp vị trí chấm + luật đầu/cuối)

```python
class Solution:
    def ambiguousCoordinates(self, s: str) -> List[str]:
        def f(i, j):                                   # mọi số hợp lệ từ s[i:j]
            res = []
            for k in range(1, j - i + 1):
                l, r = s[i : i + k], s[i + k : j]      # phần nguyên, phần thập phân (có thể rỗng)
                if (l == "0" or not l.startswith("0")) and not r.endswith("0"):
                    res.append(l + ("." if k < j - i else "") + r)
            return res

        n = len(s)
        return [f"({x}, {y})" for i in range(2, n - 1) for x in f(1, i) for y in f(i, n - 1)]
```

Công thức nhớ nhanh: số hợp lệ = phần nguyên (`"0"` hoặc không bắt đầu bằng `0`) + tùy chọn `.` + phần thập phân (không kết thúc bằng `0`); tọa độ = `"(" + x + ", " + y + ")"`.

### Checklist trước khi submit

- [ ] Phần nguyên: `"0"` hoặc không bắt đầu bằng `0`?
- [ ] Phần thập phân (nếu có): không kết thúc bằng `0`?
- [ ] Có cả trường hợp **không có dấu chấm** (số nguyên)?
- [ ] Dấu phẩy chỉ ở `m - 1` khe giữa các chữ số (mỗi vế ít nhất một chữ số)?
- [ ] Định dạng đúng `"(x, y)"` với **một khoảng trắng** sau dấu phẩy?
- [ ] Khi test, so sánh **sau khi sắp xếp** và kiểm tra không trùng?
- [ ] Đã thử `"(00)"`, `"(0000)"`, `"(100)"`, `"(00011)"`, chuỗi 10 chữ số?
- [ ] Tên method đúng `ambiguousCoordinates(self, s)` và trả `List[str]`?